# Split YOLO Dataset

Takes a flat YOLO export (`images/`, `labels/`, `data.yml`) and splits it into
`train` / `val` / `test` subfolders. Optionally excludes selected classes —
their annotation lines are dropped and remaining class ids are remapped to be
contiguous (0..N-1) in the output `data.yml`.

In [ ]:
import pathlib
import random
import shutil

import yaml

## Config

In [ ]:
SOURCE_DIR = pathlib.Path("/home/ruff/fhnw_code/load-simpla-annotations/data/yolo-format")
OUTPUT_DIR = pathlib.Path("/home/ruff/fhnw_code/load-simpla-annotations/data/yolo-split")

# Classes to drop entirely — their annotation lines are removed and remaining
# class ids are remapped to stay contiguous (0..N-1).
# Example: EXCLUDE_CLASSES = {"Container", "Wagon"}
EXCLUDE_CLASSES = {"Container", "Wagon"}

# Fractions must sum to 1.0
TRAIN_FRAC = 0.8
VAL_FRAC = 0.1
TEST_FRAC = 0.1

# Keep images with no remaining annotations after exclusion (useful as
# background/negative examples). Set to False to drop them entirely.
KEEP_EMPTY_IMAGES = True

SEED = 42

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9, "Split fractions must sum to 1.0"

In [ ]:
# Load source class names and build old_id -> new_id remap, dropping excluded classes
with open(SOURCE_DIR / "data.yml") as f:
    source_cfg = yaml.safe_load(f)

source_names: dict[int, str] = {int(k): v for k, v in source_cfg["names"].items()}

unknown = EXCLUDE_CLASSES - set(source_names.values())
if unknown:
    print(f"Warning: unknown class names in EXCLUDE_CLASSES ignored: {unknown}")

kept_old_ids = sorted(cid for cid, name in source_names.items() if name not in EXCLUDE_CLASSES)
old_to_new_id = {old_id: new_id for new_id, old_id in enumerate(kept_old_ids)}
new_names = {new_id: source_names[old_id] for old_id, new_id in old_to_new_id.items()}

print(f"Keeping {len(new_names)} of {len(source_names)} classes:")
for new_id, name in new_names.items():
    print(f"  {new_id:2d}: {name}")

In [ ]:
def remap_label_file(label_path: pathlib.Path) -> list[str]:
    """Read a YOLO label file, drop excluded classes, remap remaining ids. Returns output lines."""
    lines = []
    if not label_path.exists():
        return lines
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        old_id = int(parts[0])
        if old_id not in old_to_new_id:
            continue
        lines.append(f"{old_to_new_id[old_id]} {' '.join(parts[1:])}")
    return lines


image_dir = SOURCE_DIR / "images"
label_dir = SOURCE_DIR / "labels"

image_paths = sorted(image_dir.glob("*.jpg")) + sorted(image_dir.glob("*.png"))

pairs = []  # (image_path, output_label_lines)
for img_path in image_paths:
    label_lines = remap_label_file(label_dir / (img_path.stem + ".txt"))
    if not label_lines and not KEEP_EMPTY_IMAGES:
        continue
    pairs.append((img_path, label_lines))

print(f"{len(pairs)} of {len(image_paths)} images kept after exclusion filtering")

In [ ]:
rng = random.Random(SEED)
shuffled = pairs[:]
rng.shuffle(shuffled)

n_train = round(len(shuffled) * TRAIN_FRAC)
n_val = round(len(shuffled) * VAL_FRAC)

splits = {
    "train": shuffled[:n_train],
    "val": shuffled[n_train : n_train + n_val],
    "test": shuffled[n_train + n_val :],
}

for split_name, split_pairs in splits.items():
    out_image_dir = OUTPUT_DIR / "images" / split_name
    out_label_dir = OUTPUT_DIR / "labels" / split_name
    out_image_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok=True)

    for img_path, label_lines in split_pairs:
        shutil.copy2(img_path, out_image_dir / img_path.name)
        (out_label_dir / (img_path.stem + ".txt")).write_text(
            "\n".join(label_lines) + ("\n" if label_lines else "")
        )

    print(f"{split_name}: {len(split_pairs)} images")

In [ ]:
data_yaml = {
    "path": str(OUTPUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": new_names,
}

with open(OUTPUT_DIR / "data.yml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print(f"Wrote {OUTPUT_DIR / 'data.yml'}")

In [ ]:
# Self-check: splits are disjoint, cover all kept images, and no excluded class ids leaked through
all_split_stems = [img.stem for split_pairs in splits.values() for img, _ in split_pairs]
assert len(all_split_stems) == len(set(all_split_stems)), "An image ended up in multiple splits"
assert len(all_split_stems) == len(pairs), "Split sizes don't add up to the kept image count"

for split_name in splits:
    for label_file in (OUTPUT_DIR / "labels" / split_name).glob("*.txt"):
        for line in label_file.read_text().splitlines():
            class_id = int(line.split()[0])
            assert class_id in new_names, f"Unexpected class id {class_id} in {label_file}"

print("All checks passed.")